In [1]:
from collections import defaultdict, Counter
import math
class KneserNey5GramLM:
    def __init__(self, discount=0.75):
        self.discount = discount

        self.ngram_counts = {
            1: Counter(),
            2: Counter(),
            3: Counter(),
            4: Counter(),
            5: Counter()
        }

        self.context_counts = {
            1: Counter(),
            2: Counter(),
            3: Counter(),
            4: Counter()
        }
        self.vocab = set()

        self.continuation_counts = defaultdict(set)
    def tokenize(self, text):
        return text.lower().split()
    def train(self, corpus):
        for sentence in corpus:
            words = ["<s>"]*4 + self.tokenize(sentence) + ["</s>"]
            for word in words:
                self.vocab.add(word)
            for n in range(1, 6):
                for i in range(len(words)-n+1):
                    ngram = tuple(words[i:i+n])
                    self.ngram_counts[n][ngram] += 1
                    if n > 1:
                        context = ngram[:-1]
                        self.context_counts[n-1][context] += 1
                        word = ngram[-1]
                        self.continuation_counts[word].add(context)
        self.total_unique_bigrams = len(self.ngram_counts[2])

    def continuation_probability(self, word):

        unique_contexts = len(self.continuation_counts[word])

        if self.total_unique_bigrams == 0:
            return 1 / max(len(self.vocab),1)

        return unique_contexts / self.total_unique_bigrams

    def kneser_ney_prob(self, ngram):

        n = len(ngram)

        if n == 1:
            return self.continuation_probability(ngram[0])

        context = ngram[:-1]
        word = ngram[-1]

        count_ngram = self.ngram_counts[n][ngram]
        count_context = self.context_counts[n-1][context]

        if count_context == 0:
            return self.kneser_ney_prob(ngram[1:])

        discounted = max(count_ngram - self.discount, 0) / count_context

        unique_followers = len(
            set(
                ng[-1]
                for ng in self.ngram_counts[n]
                if ng[:-1] == context
            )
        )
        lambda_weight = (
            self.discount * unique_followers
        ) / count_context
        backoff_prob = self.kneser_ney_prob(ngram[1:])
        return discounted + lambda_weight * backoff_prob
    def predict_next(self, query, top_k=5):
        tokens = self.tokenize(query)
        if len(tokens) >= 4:
            context = tuple(tokens[-4:])
        else:
            context = tuple(tokens)
        scores = []
        for candidate in self.vocab:
            if candidate in ["<s>", "</s>"]:
                continue
            full_ngram = context + (candidate,)
            if len(full_ngram) > 5:
                full_ngram = full_ngram[-5:]
            prob = self.kneser_ney_prob(full_ngram)
            scores.append((candidate, prob))
        scores.sort(key=lambda x: x[1], reverse=True)

        return scores[:top_k]